# Popularity-Based Recommender Baseline

This notebook implements and evaluates a simple popularity-based recommender.
This serves as a strong baseline that's often hard to beat for new/cold users.

In [1]:
import sys
sys.path.append('..')  # Add parent directory to path

import pandas as pd
import numpy as np

from utils.train_test_split import chronological_train_test_split
from models.popularity_recommender import PopularityRecommender
from utils.evaluation import evaluate_model

## 1. Load Data

In [2]:
# Load the dataset
df = pd.read_csv("../data/data_with_additional_features.csv")

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Shows: {df['show_id'].nunique():,}")
df.head()

Dataset shape: (12941230, 30)
Date range: 2026-06-01 00:00:00 to 2026-06-30 23:59:00
Users: 100,000
Shows: 13,349


,user_id,playback_session_id,show_id,asset_type,episode_id,day,time,watch_minutes,timestamp,user_total_mins,...,show_median_episode_progression,show_primetime_views,show_global_popularity_log,show_primetime_ratio,show_velocity_delta,seq_decay_affinity_score,seq_immediate_recency_rank,seq_is_favorite_anchor,seq_pop_interaction_leverage,context_primetime_match_score
0,1,1,1,VOD,1.0,3,20:16,54,2026-06-03 20:16:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
1,1,2,1,VOD,2.0,3,11:05,53,2026-06-03 11:05:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
2,1,3,1,VOD,3.0,3,22:46,44,2026-06-03 22:46:00,9539.0,...,12073.0,2035.0,8.980046,0.256200,0.227082,2.224018,0,0,19.971787,0.050535
3,1,4,2,VOD,4.0,1,22:56,1,2026-06-01 22:56:00,9539.0,...,67263.0,71.0,5.673323,0.243986,1.173913,0.036882,0,0,0.209244,0.048126
4,1,5,3,RECORDING,5.0,6,15:26,27,2026-06-06 15:26:00,9539.0,...,23335.0,1058.0,9.083075,0.120159,-0.030859,0.911249,0,0,8.276939,0.023701


## 2. Chronological Train/Validation/Test Split

In [3]:
# Split data chronologically: train / validation / test
train_df, val_df, test_df = chronological_train_test_split(
    df,
    timestamp_col='timestamp',
    test_days=5,
    val_days=5,
    return_splits='train_val_test'
)

Chronological Split Summary:
  Train:      2026-06-01 00:00:00 to 2026-06-20 23:59:00 (8,705,249 rows)
  Validation: 2026-06-20 23:59:00 to 2026-06-25 23:59:00 (2,086,740 rows)
  Test:       2026-06-25 23:59:00 to 2026-06-30 23:59:00 (2,149,241 rows)


## 3. Prepare Train Seen Shows Dictionary

In [4]:
# Create a dictionary of shows each user has seen in training data
# This is used to filter out already-seen items during recommendation
train_seen_dict = train_df.groupby('user_id')['show_id'].apply(set).to_dict()

print(f"Users in training with history: {len(train_seen_dict):,}")

Users in training with history: 95,415


## 4. Train Popularity Recommender

In [5]:
# Initialize and train popularity recommender
# Using top 500 candidate pool (matching original implementation)
pop_model = PopularityRecommender(candidate_pool_size=500)
pop_model.fit(train_df, item_col='show_id', user_col='user_id')

Computing popularity scores from training data...
  - Total items in catalog: 11,954
  - Candidate pool size: 500
  - Top item popularity score: 12.93


In [6]:
# Check top 20 most popular shows
top_20_shows = pop_model.get_top_items(n=20)
print("Top 20 most popular shows:")
print(top_20_shows)

Top 20 most popular shows:
[222, 72, 10, 172, 43, 150, 146, 42, 25, 286, 196, 48, 33, 193, 223, 578, 211, 184, 304, 182]


## 5. Evaluate on Validation Set

In [7]:
# Evaluate on validation set
val_metrics = evaluate_model(
    model=pop_model,
    test_ground_truth=val_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

print("Validation Metrics:")
for metric, value in val_metrics.items():
    if 'recall' in metric or 'precision' in metric:
        print(f"  {metric}: {value*100:.2f}%")
    else:
        print(f"  {metric}: {value:.4f}")

Evaluating model on 75,859 users...

  Evaluation Results (Top-10)
  Users Evaluated: 75,859
  Recall@10:    3.48%
  MRR@10:       0.0471
  Precision@10: 1.59%

Validation Metrics:
  recall@10: 3.48%
  mrr@10: 0.0471
  precision@10: 1.59%
  num_users_evaluated: 75859.0000


## 6. Evaluate on Test Set

In [8]:
# Evaluate on test set
test_metrics = evaluate_model(
    model=pop_model,
    test_ground_truth=test_df,
    train_seen_dict=train_seen_dict,
    user_col='user_id',
    item_col='show_id',
    top_n=10,
    verbose=True
)

print("Test Metrics:")
for metric, value in test_metrics.items():
    if 'recall' in metric or 'precision' in metric:
        print(f"  {metric}: {value*100:.2f}%")
    else:
        print(f"  {metric}: {value:.4f}")

Evaluating model on 77,272 users...

  Evaluation Results (Top-10)
  Users Evaluated: 77,272
  Recall@10:    3.97%
  MRR@10:       0.0537
  Precision@10: 1.82%

Test Metrics:
  recall@10: 3.97%
  mrr@10: 0.0537
  precision@10: 1.82%
  num_users_evaluated: 77272.0000
